In [245]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

In [ ]:
fishing_ds = xr.open_dataset("./data/processed/dynamic/ESP_TRAWL.nc")
fishing = fishing_ds["hours"]
fishing = (fishing - fishing.mean()) / fishing.std()
fishing = fishing.fillna(0)   

mask_ds = xr.open_dataset("./data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]

temp_ds = xr.open_dataset("./data/processed/dynamic/to_surface.nc")
temp_ds = temp_ds.reset_coords("depth", drop=True)
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_bottom_ds = xr.open_dataset("./data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

chl_ds = xr.open_dataset("./data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

mixed_ds = xr.open_dataset("./data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

depth_ds = xr.open_dataset("./data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)

temp, temp_bottom, chl, mixed, fishing, mask, depth= xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, join="inner")


#chanels
svars = 1
dvars = 4
time_steps = 6
in_channels= dvars+svars


In [247]:
class FishingDataset(Dataset):
    def __init__(self, fishing, temp, temp_bottom, chl, mixed, mask, depth, time_steps=time_steps):
        # dynamic
        self.time_steps = time_steps
        self.fishing = fishing
        self.temp = temp
        self.chl = chl
        self.temp_bottom = temp_bottom
        self.mixed = mixed
        # static
        self.mask = mask.values.astype(np.float32)  # shape (H, W)
        self.depth = depth.values.astype(np.float32)  # shape (H, W)
        self.times = fishing.time.values

    def __len__(self):
        return len(self.times) - self.time_steps

    def __getitem__(self, idx):
        t0 = idx
        t1 = idx + self.time_steps

        # dynamic sequences: (time_steps, H, W)
        temp_seq = self.temp.isel(time=slice(t0, t1)).values.astype(np.float32)
        chl_seq  = self.chl.isel(time=slice(t0, t1)).values.astype(np.float32)
        temp_bottom_seq  = self.temp_bottom.isel(time=slice(t0, t1)).values.astype(np.float32)
        mixed_seq = self.mixed.isel(time=slice(t0, t1)).values.astype(np.float32)

        # stack dynamic channels
        x_dyn = np.stack([temp_seq, chl_seq, temp_bottom_seq, mixed_seq], axis=1)

        #static chanels
        depth_ch = self.depth[np.newaxis, np.newaxis, ...]  # (1,1,H,W)
        depth_ch = np.repeat(depth_ch, self.time_steps, axis=0)  # (T,1,H,W)

        x = np.concatenate([x_dyn, depth_ch], axis=1)  # (T, C+1, H, W)

        # target and mask -> (1, H, W)
        y = self.fishing.isel(time=t1).values.astype(np.float32)[np.newaxis, ...]
        m = self.mask[np.newaxis, ...]
        
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(m)



dataset = FishingDataset(fishing, temp, temp_bottom, chl, mixed, mask, depth, time_steps=time_steps)
n = len(dataset)

total_sequences = len(dataset)
train_end = int(0.7 * total_sequences)  # First 70% of time
val_end = train_end + int(0.15 * total_sequences)  # Next 15%

train_dataset = torch.utils.data.Subset(dataset, range(0, train_end))
val_dataset = torch.utils.data.Subset(dataset, range(train_end, val_end))
test_dataset = torch.utils.data.Subset(dataset, range(val_end, total_sequences))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [248]:

class CNN3D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(16, 1, kernel_size=1)
        )

    def forward(self, x):
        # x: (B, T, C, H, W)

        # reorder to (B, C, T, H, W)
        x = x.permute(0, 2, 1, 3, 4)

        out = self.net(x)

        # take last timestep
        out = out[:, :, -1, :, :]  # (B, 1, H, W)

        return out

In [249]:
def masked_mse_loss(pred, target, mask):
    mask = mask.unsqueeze(1)  # match channel dimension
    loss = (pred - target) ** 2
    loss = loss * mask
    return loss.sum() / mask.sum()

In [250]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_channels = 5  # temp, chl, temp_bottom, mixed, depth
model = CNN3D(in_channels=in_channels)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)



best_val_r2 = -float("inf")
best_train_r2 = -float("inf")

patience = 5
counter = 0

for epoch in range(50):
    model.train()
    total_loss = 0
    train_preds, train_targets = [], []

    for x, y, m in train_loader:
        x = x.to(device)
        y = y.to(device)
        m = m.to(device)


        pred = model(x)

        loss = masked_mse_loss(pred, y, m)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        mask_flat = m.detach().cpu().view(-1) > 0
        train_preds.append(pred.detach().cpu().view(-1)[mask_flat])
        train_targets.append(y.detach().cpu().view(-1)[mask_flat])
    
    # compute R2
    r2 = r2_score(torch.cat(train_targets).numpy(), torch.cat(train_preds).numpy())



    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for x, y, m in val_loader:
            x, y = x.to(device), y.to(device)
            m = m.to(device)
            pred = model(x)

            mask_flat = m.cpu().view(-1) > 0

            val_preds.append(pred.cpu().view(-1)[mask_flat])
            val_targets.append(y.cpu().view(-1)[mask_flat])

    val_r2 = r2_score(torch.cat(val_targets).numpy(), torch.cat(val_preds).numpy())

    if val_r2 > best_val_r2:
        counter = 0
        best_val_r2 = val_r2
    if r2 > best_train_r2:
        counter=0
        best_train_r2 = r2
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

    print(f"Epoch {epoch}, Loss: {total_loss/len(train_loader)}, R2: {r2}, Val R2: {val_r2}")

Epoch 0, Loss: 0.07245819476433098, R2: -0.06965076923370361, Val R2: -0.0015774965286254883
Epoch 1, Loss: 0.06828040432184934, R2: -0.008099913597106934, Val R2: -0.0004191398620605469
Epoch 2, Loss: 0.06802954350598156, R2: -0.004508852958679199, Val R2: -0.002211928367614746
Epoch 3, Loss: 0.06776350606232881, R2: -0.0007761716842651367, Val R2: 0.0006107091903686523
Epoch 4, Loss: 0.06774139873683453, R2: -0.00017392635345458984, Val R2: -0.00021207332611083984
Epoch 5, Loss: 0.06769011205993593, R2: 0.00044077634811401367, Val R2: 0.0007747411727905273
Epoch 6, Loss: 0.0676836048439145, R2: 0.0006282925605773926, Val R2: 0.0005183815956115723
Epoch 7, Loss: 0.06766216017305851, R2: 0.0009088516235351562, Val R2: 0.0009855031967163086
Epoch 8, Loss: 0.06765342123806477, R2: 0.0010741353034973145, Val R2: 0.0010089874267578125
Epoch 9, Loss: 0.0676400774344802, R2: 0.0012634992599487305, Val R2: 0.0012801289558410645
Epoch 10, Loss: 0.06763083571568132, R2: 0.0014167428016662598, V